# 04 - Feature Engineering


## Step 1: Import Libraries and Load Data

In [1]:
# Import libraries
import pandas as pd
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load cleaned dataset
df = pd.read_csv('cleaned_data.csv')
print(f"Dataset loaded. Shape: {df.shape}")
print("\n")
print("First few rows:")
df.head()

Dataset loaded. Shape: (100000, 25)


First few rows:


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,Order_Time,Delivery_Time_Min,Distance_km,Order_Value,Discount_Applied,Final_Amount,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin
0,ORD000001,CUST6948,19,Male,Hyderabad,Central,RES936,Restaurant_29,Chinese,10/20/2024,0:00,187.0,15.75,NaN,NaN,NaN,UPI,Delivered,NaN,DP563,5,4.4,Weekend,True,0.13
1,ORD000002,CUST6515,39,Female,Chennai,North,RES689,Restaurant_419,Chinese,8/12/2024,0:00,20.0,29.50,4869.0,20.0,4849.0,COD,Delivered,NaN,DP369,5,4.7,Weekday,True,0.48
2,ORD000003,CUST1765,39,Male,Delhi,South,RES723,Restaurant_244,Arabian,12/8/2024,0:00,207.0,9.97,757.0,20.0,737.0,Wallet,Delivered,NaN,DP580,4,4.9,Weekend,True,0.08
3,ORD000004,CUST2744,39,Male,Mumbai,Central,RES951,Restaurant_178,Chinese,10/8/2024,0:00,143.0,15.68,0.0,0.0,0.0,UPI,Cancelled,Late Delivery,DP155,0,3.4,Weekday,True,0.04
4,ORD000005,CUST4389,57,Female,Chennai,South,RES419,Restaurant_262,Chinese,2/4/2024,0:00,51.0,9.60,372.0,20.0,352.0,Card,Delivered,NaN,DP728,2,4.4,Weekend,False,0.12


## Feature 1: Order Day Type



In [3]:
# Check if Order_Day column exists
if 'Order_Day' in df.columns:
    print("Order_Day column already exists!")
    print("\n")
    print("Unique values:")
    print(df['Order_Day'].unique())
else:
    # If not, create it from date
    df['Order_Date'] = pd.to_datetime(df['Order_Date'])
    df['Order_Day'] = df['Order_Date'].dt.dayofweek
    
    # Convert to Weekday/Weekend
    # 0-4 = Monday to Friday (Weekday), 5-6 = Saturday-Sunday (Weekend)
    df['Order_Day'] = df['Order_Day'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')
    
    print("Order_Day feature created!")

print("\n")
print("Distribution:")
print(df['Order_Day'].value_counts())

Order_Day column already exists!


Unique values:
['Weekend' 'Weekday']


Distribution:
Order_Day
Weekday    71370
Weekend    28630
Name: count, dtype: int64


## Feature 2: Peak Hour Indicator

In [4]:
# Check if Peak_Hour already exists
if 'Peak_Hour' in df.columns:
    print("Peak_Hour column already exists!")
    print("\n")
    print("Distribution:")
    print(df['Peak_Hour'].value_counts())
else:
    # Create Peak_Hour from Order_Time if needed
    # Convert Order_Time to datetime and extract hour
    df['Order_Hour'] = pd.to_datetime(df['Order_Time']).dt.hour
    
    # Define peak hours: 12-2 PM (lunch) and 7-9 PM (dinner)
    df['Peak_Hour'] = df['Order_Hour'].apply(
        lambda x: True if (12 <= x <= 14) or (19 <= x <= 21) else False
    )
    
    print("Peak_Hour feature created!")
    print("\n")
    print("Distribution:")
    print(df['Peak_Hour'].value_counts())

Peak_Hour column already exists!


Distribution:
Peak_Hour
True     66415
False    33585
Name: count, dtype: int64


## Feature 3: Profit Margin Percentage


In [5]:
# Create Profit_Margin_Percentage
# Multiply by 100 to convert to percentage
df['Profit_Margin_Pct'] = df['Profit_Margin'] * 100

print("Profit_Margin_Pct feature created!")
print("\n")
print("Summary statistics:")
print(df['Profit_Margin_Pct'].describe())
print("\n")

# Show sample values
print("Sample comparison:")
print(df[['Order_ID', 'Profit_Margin', 'Profit_Margin_Pct']].head(10))

Profit_Margin_Pct feature created!


Summary statistics:
count    100000.000000
mean         15.036200
std          20.188757
min         -20.000000
25%          -2.000000
50%          15.000000
75%          32.000000
max          50.000000
Name: Profit_Margin_Pct, dtype: float64


Sample comparison:
    Order_ID  Profit_Margin  Profit_Margin_Pct
0  ORD000001           0.13               13.0
1  ORD000002           0.48               48.0
2  ORD000003           0.08                8.0
3  ORD000004           0.04                4.0
4  ORD000005           0.12               12.0
5  ORD000006           0.24               24.0
6  ORD000007           0.02                2.0
7  ORD000008           0.18               18.0
8  ORD000009           0.08                8.0
9  ORD000010           0.03                3.0


## Feature 4: Delivery Performance Category

In [6]:
# Create Delivery_Performance category
# Fast: < 30 minutes
# Normal: 30-60 minutes
# Slow: > 60 minutes

def categorize_delivery(time):
    if time < 30:
        return 'Fast'
    elif time <= 60:
        return 'Normal'
    else:
        return 'Slow'

df['Delivery_Performance'] = df['Delivery_Time_Min'].apply(categorize_delivery)

print("Delivery_Performance feature created!")
print("\n")
print("Distribution:")
print(df['Delivery_Performance'].value_counts())
print("\n")

# Calculate percentage
print("Percentage distribution:")
print((df['Delivery_Performance'].value_counts() / len(df)) * 100)

Delivery_Performance feature created!


Distribution:
Delivery_Performance
Slow      73387
Normal    20013
Fast       6600
Name: count, dtype: int64


Percentage distribution:
Delivery_Performance
Slow      73.387
Normal    20.013
Fast       6.600
Name: count, dtype: float64


## Feature 5: Customer Age Groups


In [7]:
# Create Age_Group feature
# Using standard age ranges

def categorize_age(age):
    if age < 25:
        return 'Young (18-24)'
    elif age < 35:
        return 'Adult (25-34)'
    elif age < 50:
        return 'Middle-aged (35-49)'
    else:
        return 'Senior (50+)'

df['Customer_Age_Group'] = df['Customer_Age'].apply(categorize_age)

print("Customer_Age_Group feature created!")
print("\n")
print("Distribution:")
print(df['Customer_Age_Group'].value_counts())
print("\n")

# Calculate percentage
print("Percentage distribution:")
print((df['Customer_Age_Group'].value_counts() / len(df)) * 100)

Customer_Age_Group feature created!


Distribution:
Customer_Age_Group
Middle-aged (35-49)    67601
Senior (50+)           12644
Adult (25-34)          11660
Young (18-24)           8095
Name: count, dtype: int64


Percentage distribution:
Customer_Age_Group
Middle-aged (35-49)    67.601
Senior (50+)           12.644
Adult (25-34)          11.660
Young (18-24)           8.095
Name: count, dtype: float64


## Additional Features: Order Value Categories


In [8]:
# Create Order_Value_Category
# Based on percentiles

# Calculate percentiles
low_threshold = df['Final_Amount'].quantile(0.33)
high_threshold = df['Final_Amount'].quantile(0.67)

print(f"Low threshold: ₹{low_threshold:.2f}")
print(f"High threshold: ₹{high_threshold:.2f}")
print("\n")

def categorize_order_value(amount):
    if amount < low_threshold:
        return 'Low Value'
    elif amount < high_threshold:
        return 'Medium Value'
    else:
        return 'High Value'

df['Order_Value_Category'] = df['Final_Amount'].apply(categorize_order_value)

print("Order_Value_Category feature created!")
print("\n")
print("Distribution:")
print(df['Order_Value_Category'].value_counts())

Low threshold: ₹654.00
High threshold: ₹2770.00


Order_Value_Category feature created!


Distribution:
Order_Value_Category
High Value      51978
Medium Value    24383
Low Value       23639
Name: count, dtype: int64


## Summary of All Features

In [9]:
# List all new features created
new_features = [
    'Order_Day',
    'Peak_Hour',
    'Profit_Margin_Pct',
    'Delivery_Performance',
    'Customer_Age_Group',
    'Order_Value_Category'
]

print("New Features Created:")
print("\n")
for i, feature in enumerate(new_features, 1):
    print(f"{i}. {feature}")
    if feature in df.columns:
        print(f"   - Unique values: {df[feature].nunique()}")
        print(f"   - Sample: {df[feature].unique()[:5]}")
    print("")

New Features Created:


1. Order_Day
   - Unique values: 2
   - Sample: ['Weekend' 'Weekday']

2. Peak_Hour
   - Unique values: 2
   - Sample: [ True False]

3. Profit_Margin_Pct
   - Unique values: 71
   - Sample: [13. 48.  8.  4. 12.]

4. Delivery_Performance
   - Unique values: 3
   - Sample: ['Slow' 'Fast' 'Normal']

5. Customer_Age_Group
   - Unique values: 4
   - Sample: ['Young (18-24)' 'Middle-aged (35-49)' 'Senior (50+)' 'Adult (25-34)']

6. Order_Value_Category
   - Unique values: 3
   - Sample: ['High Value' 'Medium Value' 'Low Value']



## View Updated Dataset

In [10]:
# Show sample of updated dataset with new features
print("Sample of updated dataset with new features:")
print("\n")

# Select a few columns including new features
sample_columns = ['Order_ID', 'Customer_Age', 'Customer_Age_Group', 
                  'Final_Amount', 'Order_Value_Category',
                  'Delivery_Time_Min', 'Delivery_Performance',
                  'Profit_Margin', 'Profit_Margin_Pct',
                  'Order_Day', 'Peak_Hour']

df[sample_columns].head(10)

Sample of updated dataset with new features:




,Order_ID,Customer_Age,Customer_Age_Group,Final_Amount,Order_Value_Category,Delivery_Time_Min,Delivery_Performance,Profit_Margin,Profit_Margin_Pct,Order_Day,Peak_Hour
0,ORD000001,19,Young (18-24),NaN,High Value,187.0,Slow,0.13,13.0,Weekend,True
1,ORD000002,39,Middle-aged (35-49),4849.0,High Value,20.0,Fast,0.48,48.0,Weekday,True
2,ORD000003,39,Middle-aged (35-49),737.0,Medium Value,207.0,Slow,0.08,8.0,Weekend,True
3,ORD000004,39,Middle-aged (35-49),0.0,Low Value,143.0,Slow,0.04,4.0,Weekday,True
4,ORD000005,57,Senior (50+),352.0,Low Value,51.0,Normal,0.12,12.0,Weekend,False
5,ORD000006,37,Middle-aged (35-49),NaN,High Value,56.0,Normal,0.24,24.0,Weekday,True
6,ORD000007,40,Middle-aged (35-49),NaN,High Value,273.0,Slow,0.02,2.0,Weekday,True
7,ORD000008,39,Middle-aged (35-49),3486.0,High Value,120.0,Slow,0.18,18.0,Weekend,True
8,ORD000009,39,Middle-aged (35-49),425.0,Low Value,120.0,Slow,0.08,8.0,Weekend,True
9,ORD000010,39,Middle-aged (35-49),4641.0,High Value,120.0,Slow,0.03,3.0,Weekday,True


In [11]:
# Save cleaned dataset
output_file = 'Final_dataset.csv'
df.to_csv(output_file, index=False)

print("\n" + "="*60)
print("FINAL DATA SAVED")
print("="*60)
print(f"\n✓ File saved: {output_file}")
print(f"\nDataset Summary:")
print(f"  - Rows: {len(df):,}")
print(f"  - Columns: {len(df.columns)}")
print(f"  - File size: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nReady for Exploratory Data Analysis!")


FINAL DATA SAVED

✓ File saved: Final_dataset.csv

Dataset Summary:
  - Rows: 100,000
  - Columns: 29
  - File size: 103.08 MB

Ready for Exploratory Data Analysis!
